# Tutorial: A Tiny Language Model (TLM)
> Created Mar 28 2026 for the FSU Course: *Machine Learning in Physics* <br>
> Harrison B. Prosper<br>

## Introduction
The **transformer** neural network model coupled with a mechanism called **attention** [1], which we describe in detail in this notebook, revolutionized machine-based artificial intelligence (AI) and triggered the exponential rise of **large-language models** (LLMs). The seminal work on transformers [1] describes a model based on an encoder-decoder architecture. However, since 2019, 
LLM-based models such as GPT-3, GPT-4, Claude, Llama, Mistral, Gemma, Falcon, Command R, and Grok are based on **decoder-only** models. In this tutorial, we illustrate the construction of an LLM by building, from scratch, a **tiny language model** (TLM) based on a decoder-only architecture that can be trained in about an hour on a high-end GPU.
Many of the details used in this tutorial are taken from the excellent description of **transformers** in the Annotated Transformer[2]. (Note, however, the latter considers the more complex encoder-decoder model, which is no longer favored.) 

In a text-based LLM, which is what we consider here, the corpus of text that is used to train the model is broken up into **tokens**. From these tokens a set of tokens called a **vocabulary** is constructed, wherein each unique token is mapped to a unique integer.  Ideally, the vocabulary is broad enough to represent all text that the LLM is likely to encounter. The process of breaking text, or other media, into tokens is called **tokenization**. In this tutorial, we use a hand-coded tokenizer.

The model consists of three parts:

  1. The embedding layers embed the tokens and their relative positions within sequences into a vector space. A sequence of tokens is thus mapped to a point cloud in the vector space.
  1. The transformer layers[1] implement the syntactic and semantic analyses using a technique called **masked multi-head attention**.
  1. The output layer computes weights, called **logits**, one for every possible token in the vocabulary. 

The processing of a **prompt** proceeds **autoregressively**. 

  1. The prompt is sent to the model together with a separator token that indicates the end of the prompt.
  2. The model computes logits for every token in the vocabulary, which are converted to a probability distribution over the vocabulary, conditioned on the prompt and the separator token.
  3. The token with the highest probability is taken as the next token.
  4. That token is appended to the prompt and the separator tokens, which form the next input sequence into the model.
  5. The model continues until either the end-of-sequence token or the maximum number of output tokens is reached.

## Problem Statement
In this tutorial we solve the following problem: given a **prompt** $\boldsymbol{p} = f(x)$ constructed from trigonometric and hyperbolic functions, the model emits the Taylor series expansion of the function $f(x)$ to ${\cal O}(x^6)$.
In natural language translation, a word, for example $\texttt{the}$, or part of a word, for example $\texttt{ly}$, could be a token. In symbolic mathematics, a token might be a mathematical function, e.g. $\texttt{sin}$. In this tutorial the tokens are restricted to tokens that appear in simple mathematical expressions.

## Attention

When we translate from one sequence of symbols to another, for example from one natural language to another,  the meaning of the sequences is encoded in the symbols, their relative order, and the degree to which a given symbol is related to the other symbols. Consider the phrases "the white house" and "la maison blanche". In order to obtain a correct translation it is important for the model to encode the fact that "la" and "maison" are strongly related and that their order matters, and likewise for "the" and "house". It is also important for the model to encode the strong relationship between "the" and "la", between "house" and "maison", and between "white" and "blanche". Each token needs to *pay attention to* other tokens so that semantic and syntactic facts are correctly handled.

The need for the model to pay attention to relevant linguistic facts is the basis of the  [attention mechanism](https://nlp.seas.harvard.edu/annotated-transformer/). The model associates a vector to every token that captures the strength of a token's relationship to other tokens in the sequence. Since this association mechanism operates within the same sequence (that is, within the same point cloud in the vector space in which the sequence is embedded) it is referred to as **self attention**. The optimal way to implement this idea is not known, but the attention mechanism described in Ref. [1], described later in this notebook, and subsequent variants have proven to be highly effective.

## Prediction
For a vocabulary of size $m$ and a sequence of size $k$ every position in the sequence can be filled in $m$ ways. Therefore, there are $m^k$ possible sequences of which we want the most probable. Alas we have a bit of a computational problem. For example, given a sequence of size $k=85$ tokens and a vocabulary of size $m = 28$ tokens, we face the task of finding the most probable sequence from $\sim 10^{123}$ possible sequences. Even at a trillion probability calculations per second an exhaustive search would be an utterly futile undertaking because it would take far longer to complete than the current age of the universe ($\sim 4 \times 10^{17}$ s)! Obviously, we have no choice but to use a **heuristic strategy**.
The simplest such strategy, the one we shall use, is the **greedy search** in which we choose the most probable token as the next token. For deterministic applications, such as mathematics, this is the appropriate heuristic. 

**Tensor Convention**
The convention used in the Annotated Transformer[2] is followed in which the batch is the first dimension in all tensors. 

### References
  1.  [Attention is all you need](https://papers.nips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)
  1. [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)

## Installation of `mlinphysics`

### Local installation
  ```bash
      git clone https://github.com/hbprosper/mlinphysics
      cd mlinphysics
      pip install -e .
  ```

## Running on Google Colab
If on Google Colab (https://colab.research.google.com), execute cell below.

In [1]:
try:
    import google.colab
    REPO = 'hbprosper/mlinphysics'
    
    url = f"https://raw.githubusercontent.com/{REPO}/refs/heads/main"
    !wget -q {url}/clone2colab.ipynb -O clone2colab.ipynb
    %run clone2colab.ipynb
    
    IN_COLAB = True
    
except ImportError:
    
    IN_COLAB = False

In [2]:
import os, sys
import numpy as np
import importlib
import shutil

# PyTorch
import torch
import torch.nn as nn

# ML in physics module
import mlinphysics.nn as mlp
import mlinphysics.utils.data as dat
import mlinphysics.utils.monitor as mon
import mlinphysics.utils.tutorials as tut
import mlinphysics.utils.transformer as tnm

## Computational device

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nComputational device: {str(DEVICE):s}')


Computational device: cpu


In [4]:
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed(SEED)
# torch.backends.cudnn.deterministic = True

## Constants

In [5]:
short_tutorial = True

if short_tutorial:
    DATAFILE = '../data/seq2seq_series_2terms.txt' 
    MAX_SEQ_LEN = 128
    
    # model hyperparameters       
    EMB_DIM = 72    # dimension of embedding vector space
    LAYERS  = 3     # number of decoder layers
    HEADS   = 8     # number od decoder heads
    FF_DIM  = 128
    DROPOUT = 0.1
    
    # training hyperparameters
    BATCH_SIZE    = 32
    LEARNING_RATE = 16e-4
    NITERATIONS   = 800_000
    STEP          = 100
else:
    DATAFILE = '../data/seq2seq_series.txt'
    MAX_SEQ_LEN= 256

    # model hyperparameters  
    EMB_DIM= 128   # dimension of embedding vector space
    LAYERS = 4
    HEADS  = 8
    FF_DIM = 1024
    DROPOUT= 0.1
    
    BATCH_SIZE    = 64
    LEARNING_RATE = 2e-4
    NITERATIONS   = 800_000
    STEP          = 100

## Read Sequence Data

The file **seq2seq_series_2terms.txt** contains (prompt, target) pairs where the targets are the Taylor series expansions of the corresponding sources up to an error term of ${\cal O}(x^6)$ and the sources are functions built from one or two terms randomly sampled from the set `{exp, sin, cos, tan, sinh, cosh, tanh}`. Since the source sequences are reasonably simple functions it is possible to train a transformer model to predict their Taylor series expansions in about hour on a GPU. The more complicated functions in the file **seq2seq_series.txt** require more time.

In [6]:
importlib.reload(tnm)
seqdata = tnm.SequenceData(DATAFILE, max_seq_len=MAX_SEQ_LEN)
seqdata.pprint(seqdata.sequences[0])

	reading prompt/target sequences

	sample size: 14367

0


cosh(a*x)**3 + tanh(b*x)

1 + b*x - b**3*x**3/3 + 2*b**5*x**5/15 + 3*a**2*x**2/2 + 7*a**4*x**4/8 + O(x**6)


4500


exp(a*x)*cosh(h*x)**2

...

-1 - m**2*x**2/2 - m**4*x**4/24 + c*x + c**3*x**3/6 + c**5*x**5/120 + O(x**6)


build vocabulary
{'<pad>': 0, '<sos>': 1, '<eos>': 2, '<sep>': 3, ' ': 4, '(': 5, ')': 6, '*': 7, '**': 8, '+': 9, '-': 10, '/': 11, '0': 12, '1': 13, '2': 14, '3': 15, '4': 16, '5': 17, '6': 18, '7': 19, '8': 20, '9': 21, 'O(x**6)': 22, 'a': 23, 'b': 24, 'c': 25, 'cos': 26, 'cosh': 27, 'd': 28, 'exp': 29, 'f': 30, 'g': 31, 'h': 32, 'm': 33, 'n': 34, 'sin': 35, 'sinh': 36, 'tan': 37, 'tanh': 38, 'x': 39}

tokenize prompts and targets
 14000
 14000
concatenate prompts and targets
pad sequences and bracket with <sos> and <eos>

Summary
 sample size:                   12956
  avg(sequence length):          68.2
  stdv(sequence length):         20.6
 vocabulary size:                  40

Sequence
[ 1 27  5 23  7 39  6  8 15  9 38  5 24  7 39  6  3 13  9 24  7 39  9 15
  7 23  8 14  7 39  8 14 11 14 10 24  8 15  7 39  8 15 11 15  9 19  7 23
  8 16  7 39  8 16 11 20  9 14  7 24  8 17  7 39  8 17 11 13 17  9 22  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  

cosh(a*x)**3 + tanh(b*x)

Target


1 + b*x - b**3*x**3/3 + 2*b**5*x**5/15 + 3*a**2*x**2/2 + 7*a**4*x**4/8 + O(x**6)

## Configuration

In [7]:
sequences = seqdata.sequences
ndata     = len(sequences)
train_size= 12_000
test_size =    800
val_size  = ndata - train_size - test_size

In [8]:
importlib.reload(mlp)

# name of model
# -----------------------------------------
name = 'tinyLM'

# Create new configuration object
config = mlp.Config(name, dirname=name)
# ----------------------------------------
# Training configuration
# ----------------------------------------
config('train_size',  train_size) # training dataset size
config('val_size',    val_size)
config('test_size',   test_size)
config('batch_size',  BATCH_SIZE) # number of graphs / batch
config('monitor_step',STEP)       # monitor training every n (=10) iterations
config('frac', 0.01)              # save model if average loss decreases by
                                  # more than a fraction "frac"
# ----------------------------------------
# Optimizer / scheduler configuration
# ----------------------------------------
# a step comprises a given number of iterations
n = 16
config('n_steps', n)              # number of training steps
if n > 1:
    gamma = (1/16)**(1/(n-1))
else:
    gamma = 1.0
config('gamma', gamma)            # learning rate scale factor
config('n_iterations', NITERATIONS)
config('n_iters_per_step', int(config('n_iterations') / config('n_steps')))
config('base_lr', LEARNING_RATE)  # initial learning rate
# ----------------------------------------
# Data
# ----------------------------------------
config('DATAFILE',    DATAFILE)
config('MAX_SEQ_LEN', MAX_SEQ_LEN)
# ----------------------------------------
# Model specification
# ----------------------------------------
config('EMB_DIM', EMB_DIM)  # dimension of embedding vector space
config('LAYERS',  LAYERS)   # number of encoder layers
config('HEADS',   HEADS)    # number of attention heads
config('FF_DIM',  FF_DIM)   # "hidden" dimension of ff-network
config('DROPOUT', DROPOUT)

config('VOCAB_SIZE', seqdata.VOCAB_SIZE)

config('PAD', seqdata.PAD)  
config('SOS', seqdata.SOS)
config('EOS', seqdata.EOS)
config('SEP', seqdata.SEP)
config('PROMPT_MASK', False)

print('\n\tConfiguration\n')
print(config)
print(f'\nSave configuration to file {config("file/config")}\n')

config.save()


	Configuration

name: tinyLM
file:
  config: runs/tinyLM/tinyLM_config.yaml
  losses: runs/tinyLM/tinyLM_losses.csv
  script: runs/tinyLM/tinyLM_script.pth
  params: runs/tinyLM/tinyLM_params.pth
  init_params: runs/tinyLM/tinyLM_init_params.pth
  plots: runs/tinyLM/tinyLM_plots.png
train_size: 12000
val_size: 156
test_size: 800
batch_size: 32
monitor_step: 100
frac: 0.01
n_steps: 16
gamma: 0.8312378961427878
n_iterations: 800000
n_iters_per_step: 50000
base_lr: 0.0016
DATAFILE: ../data/seq2seq_series_2terms.txt
MAX_SEQ_LEN: 128
EMB_DIM: 72
LAYERS: 3
HEADS: 8
FF_DIM: 128
DROPOUT: 0.1
VOCAB_SIZE: 40
PAD: 0
SOS: 1
EOS: 2
SEP: 3
PROMPT_MASK: false


Save configuration to file runs/tinyLM/tinyLM_config.yaml



## Datasets

In [9]:
importlib.reload(dat)

train_size = config('train_size')
val_size   = config('val_size')
test_size  = config('test_size')

# training dataset (this defines the empirical risk to be minimized)
print('training data')
train_data = dat.Dataset(
    sequences, start=0, end=train_size)

# a random subset of the training data to check for overtraining
# by comparing with the empirical risk from the validation set
print('training data for validation')
train_data_val = dat.Dataset(
    sequences, start=0, end=train_size, random_sample_size=val_size)

# validation dataset (for monitoring training)
print('validation data')
val_data = dat.Dataset(
    sequences, start=train_size, end=train_size + val_size)

# test dataset
print('test data')
test_data= dat.Dataset(sequences,
                       start=train_size + val_size,
                       end=train_size + val_size + test_size)

training data
Dataset
  shape of x: torch.Size([12000, 128])

training data for validation
Dataset
  shape of x: torch.Size([156, 128])

validation data
Dataset
  shape of x: torch.Size([156, 128])

test data
Dataset
  shape of x: torch.Size([800, 128])



## DataLoaders

In [10]:
importlib.reload(dat)

print('train data loader')
train_loader = dat.DataLoader(train_data, 
                              batch_size=config('batch_size'),
                              num_iterations=config('n_iterations'))

print('train data loader for validation')
train_loader_val = dat.DataLoader(train_data_val, 
                                  batch_size=len(train_data_val))

print('validation data loader')
val_loader = dat.DataLoader(val_data, 
                            batch_size=len(val_data))

print('test data loader')
test_loader = dat.DataLoader(test_data, 
                             batch_size=1)

train data loader
DataLoader
  Number of iterations has been specified
  maxiter:          800000
  batch_size:           32
  shuffle_step:        375

train data loader for validation
DataLoader
  maxiter:               1
  batch_size:          156
  shuffle_step:          1

validation data loader
DataLoader
  maxiter:               1
  batch_size:          156
  shuffle_step:          1

test data loader
DataLoader
  maxiter:             800
  batch_size:            1
  shuffle_step:        800



## The Decoder

The **decoder** does the following:
 1. Each token in a sequence, $\boldsymbol{X}$, is encoded as a vector $\boldsymbol{t}$ in a space of $d =$ **emb_dim** dimensions. A sequence is therefore represented as a point cloud in the vector space.
 1. The position of each token is also encoded as a vector $\boldsymbol{z}$ in a vector space of the same dimension as $\boldsymbol{t}$. We can think of $\boldsymbol{z}$ as residing in the same vector space as the vector $\boldsymbol{t}$.  Both the token and position embeddings are trainable.
 1. Each token is associated with a third vector: $\boldsymbol{v} = \nu \, \boldsymbol{t} + \boldsymbol{z}$, where the scale factor $\nu$  in this tutorial is a trainable parameter. A token and its position are encoded as a point on the line defined by the vectors $\boldsymbol{t}$ and $\boldsymbol{z}$.
 2. The vectors $\boldsymbol{v}$ are processed through $N$ *decoder layers*.

Since the sequences are **padded** so that they are all of equal length, a method is needed to ensure that the pad tokens are ignored in all calculations. This is done using **masks**. Any token that needs to be ignored
is identified with a zero in the `mask`. In the masked multi-head attention calculation (see below) all such tokens are ignored by replacing values in certain calculations with $-10^{10}$ so that when a softmax operation is performed on the results of these calculations the contribution from the associated tokens is zero. (This is explained in detail later.)

A **decoder layer** does the following:

 1. The embedded sequence and its mask are passed to a masked multi-head attention layer.
 1. A residual connection and [Layer Normalization](https://arxiv.org/abs/1607.06450) is applied.
 1. A linear layer is applied.
 1. And finally a residual connection and layer normalization is applied.


**Note**: The TLM codes below are fully type annotated to help the PyTorch scripting tool avoid mistakes. We use the scripting tool (`torch.jit.script`) to  create trained model that can be saved to a file. Of course, this assumes that we continue to work within the PyTorch ecosystem.

In [11]:
class Decoder(nn.Module):
    
    def __init__(self, 
                 vocab_size : int,   # size of vocabulary
                 max_len :    int,   # maximum sequence length
                 emb_dim :    int,   # dimension of embedding vector space
                 n_layers :   int,   # number of decoder layers
                 n_heads :    int,   # number of masked attention heads
                 ff_dim :     int,   # hidden dimension of feed-forward network
                 dropout :    float, # dropout probability
                 device :     torch.device) -> None: # computational device
        
        super().__init__()

        self.device = device

        # Represent each of the 'vocab_size' tokens by a vector 
        # of size d = emb_dim. nn.Embedding "learns" a simple
        # lookup table that maps the code for each token in the
        # vocabulary to a vector.
        self.tok_embedding = nn.Embedding(vocab_size, emb_dim)

        # Represent the position of each token by a vector of 
        # size d = emb_dim.
        # 'max_len' is the maximum length of a sequence.
        self.pos_embedding = nn.Embedding(max_len, emb_dim)

        # Create decoding layers
        self.layers  = nn.ModuleList([DecoderLayer(emb_dim, 
                                                   n_heads, 
                                                   ff_dim, 
                                                   dropout, 
                                                   device)
                                     for _ in range(n_layers)])

        # Layer to map processed token vectors to logits over the vocabulary
        self.linear  = nn.Linear(emb_dim, vocab_size)

        # Randomly set to zero weights during training.
        # Dropout is thought to mitigate over-training.
        self.dropout = nn.Dropout(dropout)

        # Factor by which to scale token embedding vectors.
        # use nn.Parameter to tell PyTorch that this is a 
        # trainable parameter.
        self.nu = nn.Parameter(torch.sqrt(torch.FloatTensor([emb_dim])))
 
    def forward(self, 
                sequence : torch.Tensor, 
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- axis for attention heads        
        batch_size, seq_len = sequence.shape

        # ---------------------------------------
        # Token embedding 
        # ---------------------------------------
        token = self.tok_embedding(sequence)
        # token: [batch_size, seq_len, emb_dim]
        
        # ---------------------------------------
        # Token position embedding
        # ---------------------------------------
        # Create a row tensor, position (=z), with entries [0, 1,..., seq_len-1]
        position = torch.arange(0, seq_len)
        # position: [seq_len]
        
        # 1. Use unsqueeze(0) to add a dimension (for the batch)
        #    so that the position tensor will have shape [1, seq_len].
        position = position.unsqueeze(0)
        
        # 2. Repeat one instance of the position tensor per row 
        #    'batch_size' times so that we get:
        # position = |position|
        #            |position|
        #                :
        #            |position|
        once_per_row = 1
        position = position.repeat(batch_size, once_per_row)
        # position: [batch_size, seq_len]
        
        # 3. Send to computational device
        position = position.to(self.device)
        # position: [batch_size, seq_len]
        
        # 4. Embed position (ordinal value of token) in a vector space.
        position = self.pos_embedding(position)
        # position: [batch_size, seq_len, emb_dim]

        # 5. Represent a token and its position as a point on a
        #    line within the vector space.
        #    (Perhaps, this could be generalized using an MLP?)
        seq = self.nu * token + position
        # seq: [batch_size, trg_len, emb_dim]

        # This seems to be helpful!
        seq = self.dropout(seq)
        
        # Process the embedded sequences through a series of layers. 
        for layer in self.layers:
            seq = layer(seq, mask)
            # seq: [batch_size, seq_len, emb_dim]

        # For each token, output 'vocab_size' logits, one for each
        # token in the vocabulary. The logits will be later converted 
        # to probabilities. This is done, for example, in the CrossEntropyLoss
        logits = self.linear(seq)
        # logits: [batch_size, seq_len, vocab_size]
            
        return logits

### Multi-Head Attention Layer

This is the key innovation in transformer models. An **attention** mechanism makes one token "pay attention to" other tokens  so that complex semantics and syntax can be captured within the sequence. In the seminal paper  [Attention is all you need](https://papers.nips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf) attention is defined by the matrix expression
\begin{align}
    \texttt{Attention}(Q, K, V) & = \texttt{softmax}\left(\mu Q K^T \right) V,
\end{align}

where $Q$ is called the `query`, $K$ the `key`, $V$ the `value`, and $\mu$ in this tutorial is a trainable scale factor, which is initialized to $1/\sqrt{d}$ where
$d =$ **emb_dim** is the dimension of the vectors that represent the tokens. The authors of [Attention is all you need](https://papers.nips.cc/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf)  found that it is better to split each vector representing a token into **n_heads** smaller vectors each of size 
$$\textrm{\bf head\_dim} = d / \textrm{\bf n\_heads}.$$ 
The integer **n_heads** is the number of so-called **attention heads**. It is claimed, with moderate justification [1], that each head pays attention to different aspects of a sequence. It is certainly plausible that this splitting procedure enhances the flexibility of the model, however, at our current level of understanding of how functions with hundred of billions of parameters truly work, such claims should be taken with a liberal pinch of salt.

In self attention, the query, key, and value tensors, which are how the tokens are represented in code, are derived from the *same* tensor via separate linear transformations of that tensor (see *Attention Algorithm* below). The coefficients of the linear functions are free parameters to be determined by the training algorithm.  The number of rows in $Q$, $K$, and $V$, namely, **query_len**,  **key_len**, and **value_len**, respectively, is equal to the sequence length **seq_len**. 

We first describe the attention mechanism mathematically and then follow with an algorithmic description that closely follows 
the description in the [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/). It is to be understood that every operation described below is performed for a batch of sequences. Therefore, when we refer to a matrix we really mean a batch of matrices.

Because each vector has been split into $n\_heads$ sub-vectors, each of dimension $head\_dim$, the calculations below apply to each attention head independently. At the end, the sub-vectors from all attention heads are coalesced back into vectors within the embedding vector space. First consider the matrix product $Q K^T$ in component form, where summation over repeated indices (the Einstein convention) is implied,
\begin{align}
A_{qk} 
& = Q_{q h} \, [K^T]_{hk}, \nonumber\\
& \quad q=1,\cdots, \text{query\_len}, \,\, h = 1, \cdots, \text{head\_dim}, \,\, k = 1, \cdots, \text{key\_len} .
\end{align}
When the matrix $A$ is scaled by $\mu$ and a softmax function applied elementwise along the key dimension (here, horizontally) the result is another matrix $W$ whose row elements, by construction, sum to unity. The matrix $W$ is then multiplied by $V$ to yield the matrix of sub-vectors
\begin{align}
    \text{Attention}_{qh}  
    & = W_{qk} V_{kh}, 
\end{align}
which encodes information about the degree of association between the vectors, each associated with a token.

Since tokens are represented by vectors, it is instructive to think of the attention computation geometrically.   Each row, $i$, of $Q$, $K$, and $V$ can be regarded as the vectors $\boldsymbol{q}_i$, $\boldsymbol{k}_i$, and $\boldsymbol{v}_i$, respectively, associated with token $i$, where each vector (really sub-vector) is of dimension $head\_dim$.  Consider, for example, a sequence with **seq_len** = 2. We can write $Q$, $K$, and $V$ as

\begin{align}
Q & = \left[\begin{matrix} \boldsymbol{q}_1 \\ \boldsymbol{q}_2 \end{matrix}\right], \\
K & = \left[\begin{matrix} \boldsymbol{k}_1 \\ \boldsymbol{k}_2 \end{matrix}\right], \text{ and} \\
V & = \left[\begin{matrix} \boldsymbol{v}_1 \\ \boldsymbol{v}_2 \end{matrix}\right] .
\end{align}

Therefore, $A = Q K^T$ is the outer product matrix
\begin{align}
A & = \left[\begin{matrix} \boldsymbol{q}_1 \\ \boldsymbol{q}_2 \end{matrix}\right] 
\left[\begin{matrix} \boldsymbol{k}_1 & \boldsymbol{k}_2 \end{matrix}\right] ,
\nonumber\\
& = \left[
\begin{matrix} 
\boldsymbol{q}_1\cdot\boldsymbol{k}_1 & \boldsymbol{q}_1\cdot \boldsymbol{k}_2 \\ 
\boldsymbol{q}_2\cdot\boldsymbol{k}_1 & \boldsymbol{q}_2\cdot \boldsymbol{k}_2
\end{matrix}
\right] .
\end{align}

The matrix $A$ can be interpreted as a measure of the degree to which the $\boldsymbol{q}$ and $\boldsymbol{k}$ vectors are aligned. Presumably, the more aligned the two vectors the stronger the relationship between the  tokens they represent. Because dot products are used, the degree of alignment depends both on the angle between the vectors as well as on their magnitudes. Consequently, two vectors can be more strongly aligned than a vector's alignment with itself! 

After the scaling and softmax operations on $A$, tokens 1 and 2 become associated with vectors $\boldsymbol{w}_1 =  (w_{11}, w_{12})$ and $\boldsymbol{w}_2 =  (w_{21}, w_{22})$, respectively, where
\begin{align}
    w_{ij} & = \frac{\exp\left(\mu \boldsymbol{q}_i \cdot \boldsymbol{k}_j \right)}
    {\sum_{k = 1}^2 \exp\left(\mu \boldsymbol{q}_i \cdot \boldsymbol{k}_k\right)} .
\end{align}

These (weight) vectors lie in the line segment $[\boldsymbol{a}, \boldsymbol{b}]$ depicted in the figure below. The line segment is a simplex (here, a 1-simplex) that is embedded in a vector space of dimension **seq_len**,  where each axis corresponds to a token.  For a sequence of length $n$, the vectors $\boldsymbol{w}_i$, $i = 1,\cdots, n$ lie in the $(n-1)$-simplex.  In our example, the vector space (not to be confused with the original embedding space of dimension $d$),  is 2-dimensional with tokens 1 and 2 represented by the orthogonal unit vectors $\boldsymbol{u}_1$ and $\boldsymbol{u}_2$, respectively. 
<img src="./simplex.png" align="left" width="250px"/>
The vector associated with token, $i$, is the weighted average 
\begin{align}
    \text{Attention}_i & = w_{i1}  \boldsymbol{v}_1 + w_{i2} \boldsymbol{v}_2
\end{align}
of the so-called value vectors $\boldsymbol{v}_1$ and $\boldsymbol{v}_2$, which are embedded representations of the tokens that reside in the original embedding space.

The upshot of this construction is that vectors representing the tokens are moved about in the embedding space in such a way that their relative positions within that space encodes information about the degree and nature of the association between the tokens. 
<br clear="left"/>


### Attention Algorithm

We now describe the attention mechanism algorithmically, following closely the description in the [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/), but with some notational changes.

<img src="./transformer.png" align="left" width="250px" style="padding: 20px;"/>

#### Step 1
The self attention mechanism starts with the tensor $X$ of shape **[batch_size,seq_len,emb_dim]**,representing the embedded sequence of tokens. (Note, emb_dim is called hid_dim in the [Annotated Transformer](https://nlp.seas.harvard.edu/annotated-transformer/)). 
Three trainable linear layers, $f_V$, $f_K$, $f_Q$ are defined, each of shape **[emb_dim,emb_dim]**, which yield the  `query`, `key`, and `value` tensors
\begin{align}
    V & = f_V(\boldsymbol{X}), \\
    K & = f_K(\boldsymbol{X}), \text{ and} \\
    Q & = f_Q(\boldsymbol{X}).
\end{align}
Each tensor $Q$, $K$, and $V$ is the same shape as $X$. 

#### Step 2
Tensors $Q$, $K$, and $V$ are reshaped by first splitting the embedding dimension, emb_dim, into n_heads blocks of size $head\_dim = emb\_dim / n\_heads$ so that their shapes become **[batch_size, seq_len, n_heads, head_dim]**. 

#### Step 3
Dimensions 1 and 2 of the tensors $Q$, $K$, and $V$ are permuted (`Tensor.permute(0, 2, 1, 3)`) so that we now have **[batch_size, n_heads, seq_len, head_dim]**, that is, we now have a different matrix of shape **[seq_len, head_dim]** for each *attention head*. Tensor $K$ is further permuted (`Tensor.permute(0, 1, 3, 2)`) to shape **[batch_size, n_heads, head_dim, seq_len]** to model $K^T$. Again, think of the last two dimensions as a matrix.

#### Step 4
Tensor $A = Q K^T$ is computed using `torch.matmul(Q, K^T)`. $A$ has shape **[batch_size, n_heads, seq_len, seq_len]**. The tensor is then scaled by the trainable parameter $\mu$. A mask, of shape **[batch_size, 1, seq_len, seq_len]**, is applied to $A$ so that tokens that must be ignored (pad tokens and tokens to the *right* of a given token) contribute zero in the softmax. The softmax is applied to the last dimension of $A$, that is, "horizontally" if one considers the last two dimensions of $A$. This yields the tensor $W$ of shape **[batch_size, n_heads, seq_len, seq_len]**.

#### Step 5
$\text{Attention} = W V$ is computed, yielding a tensor of shape 
**[batch_size, n_heads, seq_len, head_dim]**.

#### Step 6
The n_heads and seq_len dimensions of `Attention` are transposed (`Tensor.permute(0, 2, 1, 3)`) to shape **[batch_size, seq_len, n_heads, head_dim]** and forced to be contiguous in memory (`contiguous()`).

#### Step 7
The n_heads sub-vectors of dimension head_dim are concatenated using `Attention.view(batch_size, seq_len, emb_dim)` to merge the attention heads into a single `MultiHeadAttention` tensor of shape **[batch_size, seq_len, emb_dim]**.

#### Step 8
Finally, the merged `MultiHeadAttention` tensor is pushed through a trainable linear layer of shape **[emb_dim, emb_dim]** to output a tensor of shape **[batch_size, seq_len, emb_dim]**. 

### Comments
It is claimed that the  algorithm above captures the notion of every token  *paying attention to* (or "attends to") the other tokens in semantically and syntactically sensible ways and that each attention head "pays attention to" a different aspect of the sequences. Again, such claims should be taken with a pinch of salt for at least two reasons.
First, it is not at all obvious that this computation aligns with our intuitive understanding of "paying attention to" and, second, the computation is nested through multiple attention layers. Therefore, whatever the attention layers are doing, it is distributed over multiple layers in a highly non-linear, non-local, way. 

It is, however, undeniable that this simple attention-based algorithm has yielded amazing results. Therefore, we concede that, in practice,  whatever is going on in the attention layers the algorithm works wonders!

In [12]:
class MultiHeadAttention(nn.Module):
    
    def __init__(self, 
                 emb_dim : int, 
                 n_heads : int, 
                 dropout : float, 
                 device  : torch.device) -> None:
        '''
    emd_dim: int    Embeding dimension (=d)
    n_heads: int    Number of attention heads
    dropout: float  Dropout probability
    device:         Computational device
        '''

        # Initialize base class nn.Module
        super().__init__()
        
        # emb_dim must be a multiple of n_heads
        assert emb_dim % n_heads == 0
        
        self.emb_dim  = emb_dim
        self.n_heads  = n_heads

        # Compute dimension of sub-vectors
        self.head_dim = emb_dim // n_heads
        
        self.Q = nn.Linear(emb_dim, emb_dim) # Query
        self.K = nn.Linear(emb_dim, emb_dim) # Key
        self.V = nn.Linear(emb_dim, emb_dim) # Value
        
        self.O = nn.Linear(emb_dim, emb_dim) # Output
        
        self.dropout  = nn.Dropout(dropout)

        # Trainable scale factor
        self.mu = nn.Parameter(1/torch.sqrt(torch.FloatTensor([emb_dim])))
 
    def forward(self, 
                sequence : torch.Tensor, 
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len, emb_dim]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- heads dimension
        
        batch_size, seq_len, emb_dim = sequence.shape
        assert emb_dim == self.emb_dim

        # Query
        Q = self.Q(sequence)
        # Q: [batch_size, seq_len, emb_dim]

        # Key
        K = self.K(sequence)
        # K: [batch_size, seq_len, emb_dim]

        # Value
        V = self.V(sequence)
        # V: [batch_size, seq_len, emb_dim]
        
        # Split vectors of size emb_dim into 'n_heads' sub-vectors each
        # of size 'head_dim' and then permute dimensions 1 and 2
        Q = Q.view(
            batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # Q: [batch_size, n_heads, seq_len, head_dim]
        
        K = K.view(
            batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # K: [batch_size, n_heads, seq_len, head_dim]        
        
        V = V.view(
            batch_size, -1, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        # V: [batch_size, n_heads, seq_len, head_dim]
          
        # Transpose K (by permuting seq_len and head_dim)
        K = K.permute(0, 1, 3, 2)
        # K: [batch_size, n_heads, head_dim, seq_len]
        
        # Then compute mu * QK^T
        # QK^T = Q[:, :, seq_len, head_dim] * K[:, :, head_dim, seq_len]
        A = self.mu * torch.matmul(Q, K)
        # A: [batch_size, n_heads, seq_len, seq_len]

        # Apply a mask to ensure:
        #  1. a given token can pay attention only to itself
        #     and previous tokens in the sequence. That is, 
        #     subsequent tokens must be ignored. This ensures 
        #     that for every sub-sequence the algorithm is 
        #     forced to predict the next token, which, of course, 
        #     is what we want.
        #  2. pad tokens are also ignored.
        #
        # Subsequent tokens and pads are identified by zeros within
        # the mask. Everywhere there is a 0 in the mask, we replace
        # the corresponding element in the tensor A with a very 
        # large negative number so that when the softmax is applied 
        # the contribution of the associated tokens will be zero.
        A = A.masked_fill(mask == 0, -1e10)
        
        # Apply softmax to the last dimension, that is, horizontally
        # if we think about the last two dimensions.
        # WARNING: W is referred to as 'attention' in the Annotated 
        # Transformer!
        W = torch.softmax(A, dim=-1)     
        # W: [batch_size, n_heads, seq_len, seq_len]
        
        # This may be helpful
        W = self.dropout(W)
        
        # Finally, compute attention: WV
        #   W: [batch_size, n_heads, seq_len, seq_len]
        #   V: [batch_size, n_heads, seq_len, head_dim]
        attention = torch.matmul(W, V)
        # attention: [batch_size, n_heads, seq_len, head_dim]
        
        # Permute n_heads and seq_len and make sure the tensor 
        # is contiguous in memory because we're going to
        # merge (concatenate) the last two dimensions.
        attention = attention.permute(0, 2, 1, 3).contiguous()
        # attention: [batch_size, seq_len, n_heads, head_dim]
        
        # Concatenate the n heads into a single multi-head 
        # attention tensor.
        attention = attention.view(batch_size, seq_len, self.emb_dim)
        # attention: [batch_size, seq_len, emb_dim]
        
        logits = self.O(attention)
        # logits: [batch_size, seq_len, emb_dim]

        return logits

### Feedforward Layer

In [13]:
class Feedforward(nn.Module):
    
    def __init__(self, 
                 emb_dim : int, 
                 ff_dim  : int, 
                 dropout : float) -> None:
        '''
    emd_dim: int    Embeding dimension (=d)
    ff_dim: int     Number of nodes in hidden layer
    dropout: float  Dropout probability
        '''
       
        super().__init__()
        
        self.linear_1 = nn.Linear(emb_dim, ff_dim)
        
        self.linear_2 = nn.Linear(ff_dim, emb_dim)
        
        self.dropout  = nn.Dropout(dropout)

        self.silu     = nn.SiLU()
        
    def forward(self, x : torch.Tensor) -> torch.Tensor:
        # x: [batch_size, seq_len, emb_dim]
        
        x = self.linear_1(x)
        # x: [batch_size, seq_len, ff_dim]
        
        x = self.silu(x)
        
        x = self.dropout(x)
        
        x = self.linear_2(x)
        # x: [batch_size, seq_len, emb_dim]
        
        return x

### Decoder Layer

Each decoder layer has a multi-head self attention layer.

In [14]:
class DecoderLayer(nn.Module):
    
    def __init__(self, 
                 emb_dim : int, 
                 n_heads : int, 
                 ff_dim  : int, 
                 dropout : float, 
                 device  : torch.device) -> None:
        '''
    emd_dim: int    Embeding dimension (=d)
    n_heads: int    Number of attention heads
    ff_dim:  int    Number of nodes in hidden layer
    dropout: float  Dropout probability
    device:         Computational device
        '''

        super().__init__()

        # Attention mechanism
        self.self_attention      = MultiHeadAttention(
            emb_dim, n_heads, dropout, device)
        
        self.self_attention_norm = nn.LayerNorm(emb_dim)
        
        self.feedforward         = Feedforward(emb_dim, ff_dim, dropout)
        
        self.feedforward_norm    = nn.LayerNorm(emb_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, 
                sequence : torch.Tensor, 
                mask : torch.Tensor) -> torch.Tensor:
        # sequence : [batch_size, seq_len, emb_dim]
        # mask     : [batch_size, 1, seq_len, seq_len]
        #                         ^---- heads dimension
        
        # Compute attention over sequences.
        # Distinguish between sequence and seq_, since the former 
        # is needed later for residual connections.            
        seq_ = self.self_attention(sequence, mask)
        # seq_: [batch_size, seq_len, emb_dim]

        # This could be helpful
        seq_ = self.dropout(seq_)
        
        # Add residual connection and layer norm 
        seq  = self.self_attention_norm(sequence + seq_)
        # seq: [batch_size, seq_len, emb_dim]

        # Again distinguish between seq and seq_
        seq_ = self.feedforward(seq)
        # seq_: [batch_size, seq_len, emb_dim]

        # Randomly drop some elements in seq_
        seq_ = self.dropout(seq_)
        
        # Add another residual connection and layer norm
        seq  = self.feedforward_norm(seq + seq_)
        # seq: [batch_size, seq_len, emb_dim]
        
        return seq

## The `transformer` Model

The `transformer` decoder-only model, which is typical of LLMs circa 2026, encapsulates the decoder and the creation of 
the pad and subsequent (aka causal) masks. Both masks are described below.

In the pad mask a `<pad>` token is identified with a 0 and other tokens with a 1. The mask is reshaped so that it can be broadcast to tensors of shape **[batch_size, n_heads, seq_len, seq_len]** which appear in the masked multi-head attention calculation. This is necessary so that the same mask can be applied to every attention head. The mask is used to identify which tokens need to be ignored, namely, those for which the mask element is 0.

Consider a sequence $\boldsymbol{X} = \text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1,\cdots, t_{k}, \text{<eos>}$, where $\boldsymbol{p}$ denotes the sequence of prompt tokens and $t_i$ are the tokens to be predicted. The tokens $\text{<sos>}$, $\text{<sep>}$,  and $\text{<eos>}$, are the start-of-sequence, separator,  and end-of-sequence tokens, respectively. During training, ideally, for every sub-sequence, $\boldsymbol{X}_i$, we would like to predict the next token and test the quality of the prediction for all sub-sequences *simultaneously*. For example, given sub-sequences $\text{<sos>}, \boldsymbol{p}, \text{<sep>}$ and
$\text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1$, 
we would like to check simultaneously the predictions $\text{<sos>}, \boldsymbol{p}, \text{<sep>} \rightarrow y_1$ and $\text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1 \rightarrow y_2$, where $y_1$ and $y_2$ are the model predictions for the next token associated with each sub-sequence and $t_1$ and $t_2$ are the corresponding correct target tokens. 

In practice, the decoder emits vectors of weights, $\ell_i$, called **logits**, of dimension equal to the size $|\mathbb{V}|$ of the vocabulary, $\mathbb{V}$, which are converted to a discrete conditional probability distribution 
\begin{align}
p(t_{i+1} \in \mathbb{V}| \boldsymbol{X}_i).
\end{align}
The most probable token is taken to be the next output token, $y_i$.

A crucial advantage of a transformer compared with earlier sequence models is that during training, the model computes logits for all the sub-sequences of a sequence, $\boldsymbol{X}$, *in parallel*. This is achieved with a simple, but clever, trick: the **subsequent** or **causal mask**, `sub_mask`. This mask, created using  the function $\texttt{torch.tril}$, is a lower diagonal square matrix where the elements above the diagonal are zero and all other elements are unity. Just before the application of the softmax in the attention code,  the elements identified by the zeros in `sub_mask` of the tensor that enters the softmax are replaced with a large negative number. This causes the masked tokens to contribute zero when the softmax is applied, thereby ensuring that the calculations on a sub-sequence depend only on the tokens of the sub-sequence. For a given sub-sequence, the model cannot "cheat" by looking ahead at the correct next token, that is, cannot "attend to" subsequent tokens. Since the pad tokens are also masked, they too cannot contribute to the attention calculation. 

Consider, for example, the sequence $\boldsymbol{X} = \text{<sos>}, p, \text{<sep>}, t_1, \text{<eos>}$ comprising 5 tokens.  The semi-causal mask, called `sub_mask` in the code, looks like this:

$$\begin{pmatrix}
1 & 0 & 0 & 0 & 0\\
1 & 1 & 0 & 0 & 0\\
1 & 1 & 1 & 0 & 0\\
1 & 1 & 1 & 1 & 0\\
1 & 1 & 1 & 1 & 1\\
\end{pmatrix}.$$

When applied to the sequence of tokens, $\boldsymbol{X}$, the causal (or subsequent) mask, `sub_mask`, ensures that for every token the model has access only to the token and its predecessors. In other words, for a given sub-sequence, the tokens can attend to only tokens within the sub-sequence. For example, the first row of the causal mask is **[1, 0, 0, 0, 0]**. The sub-sequence $\text{<sos>}$ can attend to itself only, so the model is forced to predict the next token given the sequence $\text{<sos>}$. (At this stage, the model would be hard-pressed to do so!) The second row of the causal mask is **[1, 1, 0, 0, 0]**. In this case, the tokens  $\text{<sos>}$ and $p$ can attend to each other but not to the subsequent tokens. Consequently, the model is forced to predict the next token. The same holds true for the remaining sub-sequences. Again it should be stressed that all of these calculations are done in parallel. 

The overall mask is the logical AND of the pad and causal masks.
Moreover, during training, the 
causal mask makes it possible to compute losses for every sub-sequence simultaneously,
\begin{align}
  \text{<sos>},\boldsymbol{p} & \rightarrow \ell_0  \rightarrow loss(\ell_0, \text{<sep>}),\\
  \text{<sos>},\boldsymbol{p}, \text{<sep>} & \rightarrow \ell_1  \rightarrow loss(\ell_1, t_1),\\
  \text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1  & \rightarrow \ell_2 \rightarrow loss(\ell_2, t_2), \\
        : & : \\
  \text{<sos>}, \boldsymbol{p}, \text{<sep>}, t_1,\cdots, t_{k}  & \rightarrow \ell_{k+1}  \rightarrow loss(\ell_{k+1}, \text{<eos>}) .
\end{align}
Notice that the losses are computed on the tokens to be predicted as well as the separator token. 
    
In evaluation mode, the model is used *autoregressively*: the prompt sequence $\text{<sos>}, \boldsymbol{p}, \text{<sep>}$ is entered into the model, which predicts the next token, $y_1$. That token is appended to the current input sequence to form the next input sequence $\text{<sos>}, \boldsymbol{p}, \text{<sep>}, y_1$ and the procedure repeats until either the token $\text{<eos>}$ is predicted or the maximum allowed output sequence length is reached, whichever comes first.

In [15]:
importlib.reload(tnm)

# Note: The decoration @torch.jit.unused will cause the PyTorch scripting tool
#       to ignore the decorated function.

class TinyLM(mlp.Model):
    
    def __init__(self, 
                 vocab_size : int,
                 max_seq_len : int,
                 emb_dim : int,
                 layers : int,
                 heads : int,
                 ff_dim : int,
                 dropout : float,
                 pad : int,
                 sos : int,
                 eos : int,
                 sep : int,
                 prompt_mask : bool=False,
                 device : torch.device=torch.device(
                     'cuda' if torch.cuda.is_available() else 'cpu')):
        '''
    Arguments:
        
         vocab_size : int            Vocabulary size
         max_seq_len : int           Maximum sequence length
         emb_dim : int               Embedding dimension
         layers : int                Number of decoding layers
         heads : int                 Number of attention heads
         ff_dim : int                Feed-forward network dimension
         dropout : float             Dropout probability
         
         pad : int                   Pad code
         sos : int                   Start-of-sequence code
         eos : int                   End-fo-sequence code
         sep : int                   Separator code
         
         prompt_mask : bool          If true allow all prompt tokens 
                                     to attend to each other [False]
         device:  torch.device       Computational device
        '''
        
        super().__init__()

        self.pad : int = pad  # make sure jit sees th
        self.sos : int = sos
        self.eos : int = eos
        self.sep : int = sep

        self.max_len : int = max_seq_len
        self.prompt_mask : bool = prompt_mask
        self.device : torch.device = device
        
        self.decoder = Decoder(
            vocab_size, max_seq_len, emb_dim, layers, heads, ff_dim, dropout,
            device)

    def make_mask(self, sequence: torch.Tensor) -> torch.Tensor:
        # sequence: [batch_size, seq_len]
        _, seq_len = sequence.shape
    
        pad_mask = (sequence != self.pad).unsqueeze(1).unsqueeze(2)
        # pad_mask: [batch_size, 1, 1, seq_len]

        sub_mask = torch.tril(
            torch.ones((seq_len, seq_len), device=self.device, dtype=torch.bool))
        # sub_mask: [seq_len, seq_len]

        # logical AND of the two masks
        mask = pad_mask & sub_mask
        # mask: [batch_size, 1, seq_len, seq_len]

        # If requested, allow all prompt tokens to attend to
        # each other by placing True at every prompt token. 
        if self.prompt_mask:
            is_sep = sequence == self.sep # find separators
            
            prompt_mask = ~torch.logical_xor(
                is_sep.cumsum(dim=-1)>=1, is_sep).unsqueeze(1).unsqueeze(2)
            
            mask = mask | prompt_mask
            
        return mask

    @torch.jit.unused
    def make_loss_mask(self, sequence: torch.Tensor) -> torch.Tensor:
        """
        Mask with 1 at positions at which loss should be computed, 
        0 elsewhere. Losses are computed for the separator token and
        all positions thereafter, but excluding the <pad> tokens.
    
        Arguments:
            sequence: [batch_size, seq_len]   A tensor of integers
    
        Returns:
            mask:     [batch_size, seq_len]   A float tensor, values in {0.0, 1.0}
        """

        # Place True at every <sep> token in the batch.
        is_sep = sequence == self.sep
        
        # Place True at <sep> and target tokens in the batch.
        sep_and_target = is_sep.cumsum(dim=-1) >= 1

        # Place True at every <pad>
        not_pad = sequence != self.pad

        # AND the two masks and convert to floats
        return (sep_and_target & not_pad).float()

    def forward(self, sequence : torch.Tensor) -> torch.Tensor:
        # sequence: [batch_size, seq_len]
       
        mask  = self.make_mask(sequence)
        # mask: [batch_size, 1, seq_len, seq_len]
        
        logits = self.decoder(sequence, mask)
        # logits: [batch_size, seq_len, vocab_size]

        return logits

    @classmethod
    @torch.jit.unused
    def from_config(cls, config: mlp.Config) -> 'TinyLM':
        """Convenience constructor for the training notebook."""
        
        return cls(
            vocab_size = config('VOCAB_SIZE'),
            max_seq_len= config('MAX_SEQ_LEN'),
            emb_dim    = config('EMB_DIM'),
            layers     = config('LAYERS'),
            heads      = config('HEADS'), 
            ff_dim     = config('FF_DIM'), 
            dropout    = config('DROPOUT'),
            pad        = config('PAD'),
            sos        = config('SOS'),
            eos        = config('EOS'),
            sep        = config('SEP'),
            prompt_mask= config('PROMPT_MASK', False),
            device     = config('DEVICE', 
                                torch.device(
                                    'cuda' if torch.cuda.is_available() \
                                    else 'cpu')) 
        )

In [16]:
def print_mask_check(sequence, mask, vocab):
    """vocab: dict mapping token id -> string token
    By Claude
    """
    for b in range(sequence.size(0)):
        tokens = [vocab[i.item()] for i in sequence[b]]
        values = [str(int(m.item())) for m in mask[b]]
        print("  ".join(f"{t:>6}" for t in tokens))
        print("  ".join(f"{v:>6}" for v in values))
        print()

## Training the Tiny Language Model

Our model is miniscule compared with the transformer models used today of which there are many variants, as noted in the Introduction. Indeed, the model is small enough to be trained on a single GPU in less than an hour!
<img src="./logits_grid.png" align="left" width="500px"/>
In Section *Transformer Model* we advertised a key feature of this model: it processes all tokens in a sequence, $\boldsymbol{X}$ in parallel and computes logits for the next token for every sub-sequence. Consider, again, the same sequence as  before of size $k = 5$ as depicted in the figure to the left. The model outputs a logit for every input token and every token in the vocabulary as illustrated by the 2D tensor on the left of the figure. The size of the vocabulary in the figure is $|\mathbb{V}|=7$. The input sequence is shown as a column to the right labeled targets. Since there's nothing to predict after `<eos>`, we slice off the `<eos>` token from the end of the logits tensor. Likewise, there's nothing to predict before `<sos>`, so we slice off that token from the input sequence. The tokens of the truncated input sequence serve as the targets. Now that the logit and target tensors are correctly aligned, we can compute the loss for each sub-sequence.
<br clear="left"/>

### Loss function
A transformer is a multi-class classifier: the logits, when converted to probabilities, are used to decide which token comes next. Like most multi-class classifiers, a transformer is trained using the **cross entropy loss**. Given the data $(\ell_i, t_i)$, where $\ell_i$ is a vector of logits and $t_i$ is the true token at the $i^\text{th}$ position, the loss is given by
\begin{align}
    loss(\ell_{i, k}, t_i) & = - \log p_k, \quad\text{ with } k = t_i, \\
        p_k & = \texttt{softmax}_k(\ell_i) \equiv \frac{\exp(\ell_{i, k})}{\sum_{j} \exp(\ell_{i,j})},
\end{align}
and $\ell_{i, k}$ denotes the $k^\text{th}$ component of the logit vector $\ell_i$. The **empirical risk**, that is, average loss, is computed over a batch of sequences for all tokens *after* the `<sep>` token in each sequence, but excluding the `<pad>` tokens.

In [17]:
def train(objective, optimizer, scheduler, monitor,
          train_loader, train_small_loader, val_loader):

    for sequence in train_loader:
        
        objective.train()
        
        R = objective(sequence)
        
        optimizer.zero_grad()     # zero gradients
        
        R.backward()              # compute gradients

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()          # make a single step in average loss

        # check whether to update learning rate
        scheduler.step()

        if monitor.step():

            # set mode to evaluation so that training-specific
            # operations such as dropout, etc., are disabled.
            objective.eval()

            seq = next(iter(train_small_loader))
            t_loss = objective(seq).item()
 
            seq = next(iter(val_loader))
            v_loss = objective(seq).item()

            # return current learning rate
            lr = scheduler.lr()

            # update loss file
            monitor(t_loss, v_loss, lr)

In [18]:
importlib.reload(tnm)

config('DEVICE', DEVICE)
model = TinyLM.from_config(config).to(DEVICE)
print(model)
print(f'The model has {mlp.number_of_parameters(model)} trainable parameters')
            
optimizer = torch.optim.Adam(model.parameters(), lr=config('base_lr'))

averageloss = nn.CrossEntropyLoss(reduction='none')
# --------------------------------------------------------
# Make a specialized objective from mlp.Objective
# --------------------------------------------------------
class TLMObjective(mlp.Objective):
    
    def __init__(self, model, avgloss):

        super().__init__(model, avgloss)
        
    def forward(self, sequence):
        # sequence[batch_size, seq_len]

        # Run the model to compute, in parallel, a vector 
        # of logits for every token in the sequence. The
        # logits for a given token are used to predict
        # the next token.
        logits = self.model(sequence)
        # logits: [batch_size, seq_len, vocab_size]
        batch_size, seq_len, vocab_size = logits.shape
        
        # Slice off the <eos> token from the logits tensor because
        # there's nothing to predict after the end-of-sequence.
        logits_  = logits[:, :-1, :]
        # logits_: [batch_size, seq_len-1, vocab_size]

        # Slice off the <sos> token from the sequence because
        # there are no logits for the start-of-sequence!
        targets  = sequence[:, 1:]
        # targets: [batch_size, seq_len-1]

        # Compute a loss for every token, but do not reduce!
        # Yes, there is wasted computation, but this makes
        # the code cleaner.
        loss_per_token = self.avgloss(
                logits_.reshape(-1, vocab_size), # Flatten
                targets.reshape(-1),             # Flatten
        ).reshape(batch_size, -1)  # Back to shape [batch_size, seq_len-1]

        # The logits and targets are now aligned. Create a mask 
        # to exclude the whole of the delimited prompt and all 
        # remaining pads after the separator token.
        loss_mask = self.model.make_loss_mask(sequence[:, :-1])

        # Use loss_mask to zero out all losses except the relevant ones. 
        # Then average to get the empirical risk, R
        R = (loss_per_token * loss_mask).sum() / loss_mask.sum()

        return R

objective = TLMObjective(model, averageloss)

# Instantiate learning rate step scheduler
scheduler = mlp.LRStepScheduler(
    optimizer,  
    n_steps=config('n_steps'), 
    n_iters_per_step=config('n_iters_per_step'), 
    base_lr=config('base_lr'), 
    gamma=config('gamma')
)

# Instantiate object that saves average losses to
# a csv file for realtime monitoring, as well as
# the model with the lowest average loss
monitor = mon.Monitor(
    config('n_iterations'),
    config('file/losses'),
    monitorstep=config('monitor_step'),
    newlossfile=True,
    frac=config('frac'),
    model=model,
    paramsfile=config('file/params')
)

TinyLM(
  (decoder): Decoder(
    (tok_embedding): Embedding(40, 72)
    (pos_embedding): Embedding(128, 72)
    (layers): ModuleList(
      (0-2): 3 x DecoderLayer(
        (self_attention): MultiHeadAttention(
          (Q): Linear(in_features=72, out_features=72, bias=True)
          (K): Linear(in_features=72, out_features=72, bias=True)
          (V): Linear(in_features=72, out_features=72, bias=True)
          (O): Linear(in_features=72, out_features=72, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (self_attention_norm): LayerNorm((72,), eps=1e-05, elementwise_affine=True)
        (feedforward): Feedforward(
          (linear_1): Linear(in_features=72, out_features=128, bias=True)
          (linear_2): Linear(in_features=128, out_features=72, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (silu): SiLU()
        )
        (feedforward_norm): LayerNorm((72,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p

In [19]:
print(f'\n\tComputational device: {str(DEVICE):s}\n')

TRAIN = False

if TRAIN:

    monitor.start()

    train(
        objective, optimizer, scheduler, monitor,
        train_loader, train_loader_val, val_loader
    )

    monitor.end()


	Computational device: cpu



## Using the Model

The test data are already tokenized, coded, and bracketed. Given a trained model and a prompt, generate output autoregressively.

In [20]:
def generate(model, 
             prompt: torch.Tensor, 
             eos: int, 
             pad: int,
             max_len: int, 
             device: torch.device) -> torch.Tensor:
    '''
    Given a model and a prompt, generate output. 
    '''

    if prompt.dim() != 2:
        prompt = prompt.view(1, -1)
    # prompt: [1, prompt_len]
    
    sequence = prompt.to(device)
    _, prompt_len = sequence.shape
    
    output = []

    for _ in range(max_len - prompt_len):

        # Execute model. The model computes logits for every
        # sub-sequence of the current sequence. But we really
        # need only the logits for the last sub-sequence, i.e.,
        # for the current sequence.
        logits    = model(sequence)
        # logits: [1, seq_len, vocab_size]
        
        # Get logits of last sub-sequence, i.e., the current sequence.
        last      = logits[0, -1, :]
        # last: [seq_len, vocab_size]

        # Convert logits to probabilities.
        probs     = torch.softmax(last, dim=-1)
        # probs: [seq_len, vocab_size]
        
        # Choose the most probable token.
        _, code_t = torch.topk(probs, k=1)
        code      = int(code_t[0].item())

        if code == pad: continue
        if code == eos: break

        # Add next token to current input sequence and iterate
        next_token = torch.full((1, 1), code, dtype=torch.long, device=device)
        sequence   = torch.cat([sequence, next_token], dim=-1)
        output.append(code)

    return torch.tensor(output, dtype=torch.long)

In [21]:
importlib.reload(mlp)
# ----------------------------------------------------------------
# Example of a model with ~99% accuracy
config_filename = 'runs/TLM/TLM_config.yaml'
#config_filename = config('file/config')

config = mlp.Config(config_filename)
print('\tCONFIGURATION\n')
print(config)

# Initialize model to the best-fit parameters
model = TinyLM.from_config(config).to(DEVICE)
model.load(config('file/params'))
print('\tMODEL\n')
print(model)
print(f'Number of parameters: {mlp.number_of_parameters(model)}')

# Save best model
print(f'\nSave fully-loaded model to {config('file/script')}')
torch.jit.script(model).save(config('file/script'))

	CONFIGURATION

name: TLM
file:
  config: runs/TLM/TLM_config.yaml
  losses: runs/TLM/TLM_losses.csv
  params: runs/TLM/TLM_params.pth
  script: runs/TLM/TLM_script.pth
  init_params: runs/TLM/TLM_init_params.pth
  plots: runs/TLM/TLM_plots.png
train_size: 12000
val_size: 156
test_size: 800
batch_size: 32
monitor_step: 100
frac: 0.01
n_steps: 16
n_iterations: 800000
n_iters_per_step: 50000
base_lr: 0.0016
gamma: 0.8312378961427878
DATAFILE: ../data/seq2seq_series_2terms.txt
MAX_SEQ_LEN: 128
EMB_DIM: 72
LAYERS: 3
HEADS: 8
FF_DIM: 128
DROPOUT: 0.1
VOCAB_SIZE: 40
PAD: 0
SOS: 1
EOS: 2
SEP: 3
PROMPT_MASK: false

	MODEL

TinyLM(
  (decoder): Decoder(
    (tok_embedding): Embedding(40, 72)
    (pos_embedding): Embedding(128, 72)
    (layers): ModuleList(
      (0-2): 3 x DecoderLayer(
        (self_attention): MultiHeadAttention(
          (Q): Linear(in_features=72, out_features=72, bias=True)
          (K): Linear(in_features=72, out_features=72, bias=True)
          (V): Linear(in_features

In [22]:
# Load saved model
model = torch.jit.load(config('file/script'))
model.eval()

PRINT_MISTAKES = False
M = 0
F = 0.0
T = 10
n_mistakes = 100
k_mistakes = 0

EOS = config('EOS')
PAD = config('PAD')
MAX_LEN = config('MAX_SEQ_LEN')

for i, sequence in enumerate(test_loader):
    # sequence: [1, seq_len]

    # Extract prompt and target
    prompt, target = seqdata.split(sequence)
    target_= seqdata.str(target)

    # Generate output sequence
    output = generate(
        model, prompt, EOS, PAD, MAX_LEN, DEVICE)

    output_= seqdata.str(output)

    # count how often we're right
    if output_ == target_:
        M += 1
        F = M / (i+1)
    else:
        if PRINT_MISTAKES:
            print()
            print(f'{i:3d} true: {target_}')
            print(f'    pred: {output_}')
            print('-'*80)
            k_mistakes += 1
            PRINT_MISTAKES = k_mistakes < n_mistakes
            
    if i % T == 0:
        print(f'\r{i:8d}\taccuracy: {F:8.3f}', end='')
        
N  = len(test_data)
dF = np.sqrt(F*(1-F)/N)
print()
print(f'Accuracy: {F:8.3f} +/- {dF:.3f}')

     790	accuracy:    0.994
Accuracy:    0.993 +/- 0.003
